In [1]:
import torch
from momentfm import MOMENTPipeline
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Load MOMENT model in classification mode
model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={
        'task_name': 'classification',
        'n_channels': 1,  # Number of input channels (e.g., ECG signal)
        'num_class': 2    # Number of classes for classification
    },
)
model.init()
model.to("cuda").float()

print(model)

/home/user/anaconda3/envs/tsfm_anamoly_paper/lib/python3.11/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/user/anaconda3/envs/tsfm_anamoly_paper/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/user/Downloads/moment/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


MOMENTPipeline(
  (normalizer): RevIN()
  (tokenizer): Patching()
  (patch_embedding): PatchEmbedding(
    (value_embedding): Linear(in_features=8, out_features=1024, bias=False)
    (position_embedding): PositionalEmbedding()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
  

In [2]:
from momentfm.data.classification_dataset import ClassificationDataset

# Load training and test datasets
train_dataset = ClassificationDataset(data_split='train')
test_dataset = ClassificationDataset(data_split='test')

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=False)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, drop_last=False)

Number of the Time Series:  500
Length of the Time Series:  672
Number of the Time Series:  500
Length of the Time Series:  672


In [3]:
# idx = np.random.randint(0, len(train_dataset))
# print(idx)

In [4]:
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Function for training the model
def train_model(model, train_dataloader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct_preds = 0
        total_preds = 0
        for batch_x, batch_masks, batch_labels in tqdm(train_dataloader, total=len(train_dataloader)):
            batch_x = batch_x.to("cuda").float()
            batch_masks = batch_masks.to("cuda")
            batch_labels = batch_labels.to("cuda")

            # Forward pass
            output = model(x_enc=batch_x, input_mask=batch_masks)
            logits = output.logits

            # Compute loss
            loss = criterion(logits, batch_labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Track accuracy
            _, predicted = torch.max(logits, 1)
            correct_preds += (predicted == batch_labels).sum().item()
            total_preds += batch_labels.size(0)

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_dataloader)
        epoch_acc = correct_preds / total_preds * 100
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

# Train the model
train_model(model, train_dataloader, criterion, optimizer, num_epochs=10)

  0%|          | 0/16 [00:00<?, ?it/s]/home/user/anaconda3/envs/tsfm_anamoly_paper/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/user/anaconda3/envs/tsfm_anamoly_paper/lib/python3.11/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|██████████| 16/16 [00:00<00:00, 19.13it/s]


Epoch 1/10, Loss: 0.6651, Accuracy: 74.00%


100%|██████████| 16/16 [00:00<00:00, 23.60it/s]


Epoch 2/10, Loss: 0.6413, Accuracy: 74.00%


100%|██████████| 16/16 [00:00<00:00, 23.58it/s]


...

Epoch 8/10, Loss: 0.5765, Accuracy: 74.00%


100%|██████████| 16/16 [00:00<00:00, 23.48it/s]


Epoch 9/10, Loss: 0.5744, Accuracy: 74.00%


100%|██████████| 16/16 [00:00<00:00, 23.53it/s]

Epoch 10/10, Loss: 0.5727, Accuracy: 74.00%


In [5]:
from sklearn.metrics import precision_score, recall_score, f1_score

def test_model(model, test_dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch_x, batch_masks, batch_labels in tqdm(test_dataloader, total=len(test_dataloader)):
            batch_x = batch_x.to("cuda").float()
            batch_masks = batch_masks.to("cuda")
            batch_labels = batch_labels.to("cuda")

            # Forward pass
            output = model(x_enc=batch_x, input_mask=batch_masks)
            logits = output.logits

            # Get predicted labels
            _, predicted = torch.max(logits, 1)
            all_preds.append(predicted.cpu().numpy())
            all_labels.append(batch_labels.cpu().numpy())

    # Concatenate results
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary')  # Use 'macro' or 'micro' for multi-class
    recall = recall_score(all_labels, all_preds, average='binary')
    f1 = f1_score(all_labels, all_preds, average='binary')

    # Print results
    print(f"Test Accuracy : {accuracy * 100:.2f}%")
    print(f"Precision     : {precision * 100:.2f}%")
    print(f"Recall        : {recall * 100:.2f}%")
    print(f"F1 Score      : {f1 * 100:.2f}%")


In [6]:
test_model(model, test_dataloader)

100%|██████████| 16/16 [00:00<00:00, 24.36it/s]

Test Accuracy : 78.80%
Precision     : 78.80%
Recall        : 100.00%
F1 Score      : 88.14%


Zeroshot

In [7]:
import torch
from momentfm import MOMENTPipeline
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import accuracy_score
from momentfm.data.classification_dataset import ClassificationDataset

# Load MOMENT model in classification mode (zero-shot: no training)
model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={
        'task_name': 'classification',
        'n_channels': 1,
        'num_class': 2
    },
)
model.init()
model.to("cuda").float()
#print(model)

# Load test dataset only
test_dataset = ClassificationDataset(data_split='test')
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, drop_last=False)

# Zero-shot evaluation: directly run inference
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch_x, batch_masks, batch_labels in test_dataloader:
        batch_x = batch_x.to("cuda").float()
        batch_masks = batch_masks.to("cuda")
        batch_labels = batch_labels.to("cuda")
        
        # Forward pass
        output = model(x_enc=batch_x, input_mask=batch_masks)
        logits = output.logits
        _, predicted = torch.max(logits, 1)
        all_preds.append(predicted.cpu().numpy())
        all_labels.append(batch_labels.cpu().numpy())

# Concatenate results
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

# Accuracy (optional): measures zero-shot performance
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='binary')  # Use 'macro' or 'micro' for multi-class
recall = recall_score(all_labels, all_preds, average='binary')
f1 = f1_score(all_labels, all_preds, average='binary')
print(f"Zero-Shot Test Accuracy: {accuracy * 100:.2f}%")
print(f"Precision     : {precision * 100:.2f}%")
print(f"Recall        : {recall * 100:.2f}%")
print(f"F1 Score      : {f1 * 100:.2f}%")

/home/user/Downloads/moment/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


Number of the Time Series:  500
Length of the Time Series:  672
Zero-Shot Test Accuracy: 67.20%
Precision     : 84.23%
Recall        : 71.83%
F1 Score      : 77.53%
